In [ ]:
import numpy as np
from tqdm import tqdm
from util.data_generate import *
from util.methods import *

results = [0,0,0]
with open('output.csv', 'w') as f:
    f.write('improvement,smallest,submodular\n')  # 可选：写入列名
#result = 0

# 1. 参数设置
p = 50
m = 5
group_sizes=[10, 10, 10, 10, 10] 
#group_sizes = [15, 10, 20, 10, 15, 10, 10, 10]
J = 10000

for seed in tqdm(range(100,200,1)):

    mu,std,R,Sigma = data1(p=p, seed=seed, group_sizes=group_sizes)
    #np.random.seed(seed)
#-----------------------------------------------------------------------------------------
    B, choosen_index = improvement(p,m,mu,std,R,Sigma)
#-----------------------------------------------------------------------------------------
    # indices_smallest = np.argsort(mu)[:m] #直接取最小值
    # sub_Sigma = Sigma[np.ix_(indices_smallest, indices_smallest)]
    # L = np.linalg.cholesky(sub_Sigma) 
#-----------------------------------------------------------------------------------------
    cluster_labels = cluster_subjects(R, n_clusters=m, method='average')
    cluster_indices = find_min_mu_in_clusters(mu, cluster_labels)
    sub_Sigma = Sigma[np.ix_(cluster_indices, cluster_indices)]
    L = np.linalg.cholesky(sub_Sigma) 
#-----------------------------------------------------------------------------------------
    U, mixed_index = mixed(p,m,mu,std,R,Sigma,J=J)
#-----------------------------------------------------------------------------------------
    G_cul = 0
    J1 = 10000
    for j in range(J1):
        z = np.random.randn(m)
        temp = mu[choosen_index] + B @ z
        G_cul += np.min(temp)
    G1 = G_cul/J1

    G_cul = 0
    for j in range(J1):
        z = np.random.randn(m)
        temp = mu[cluster_indices] + L @ z
        G_cul += np.min(temp)
    G2 = G_cul/J1 

    G_cul = 0
    for j in range(J1):
        z = np.random.randn(m)
        temp = mu[mixed_index] + U @ z
        G_cul += np.min(temp)
    G3 = G_cul/J1
    G = np.array([G1,G2,G3])
    results[np.argmin(G)]+=1

    with open('output.csv', 'a') as f: 
        line = ','.join(map(str, G)) + '\n'  # 转为逗号分隔的字符串
        f.write(line)

print(results)



  0%|          | 0/100 [00:00<?, ?it/s]d:\PHD\Project\Pilot\util\methods.py:111: ClusterWarning: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  linkage_matrix = hierarchy.linkage(distance_matrix, method=method)
100%|██████████| 100/100 [02:09<00:00,  1.29s/it]

[0, 19, 81]


In [2]:
import numpy as np
import pandas as pd
data = pd.read_csv('output.csv')
improvement = np.mean(data['improvement'].to_numpy())
smallest = np.mean(data['smallest'].to_numpy())
submodular = np.mean(data['submodular'].to_numpy())
print('improvement:%f\nsmallest:%f\nsubmodular:%f\n'%(improvement,smallest,submodular))

improvement:100.000000
smallest:64.530222
submodular:64.400044

